# Oral health and diabetes in US adults (NHANES 2013-2014)
Personal learning project: do simple oral health measures add information
to a diabetes prediction model beyond age, sex and BMI?

## 1. Download and load the data
Downloads five NHANES 2013-2014 files from the CDC (demographics, HbA1c,
body measures, dentition exam, periodontal exam) and loads them.
Participants are linked across files by SEQN, the unique participant ID.

In [14]:
import requests

files = ["DEMO_H", "GHB_H", "BMX_H", "OHXDEN_H", "OHXPER_H"]
base = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/"

for f in files:
    r = requests.get(base + f + ".xpt", headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    if r.status_code == 200:
        with open(f + ".xpt", "wb") as out:
            out.write(r.content)
        print(f, "OK", len(r.content) // 1024, "KB")
    else:
        print(f, "FAILED", r.status_code)

DEMO_H OK 3743 KB
GHB_H OK 110 KB
BMX_H OK 1997 KB
OHXDEN_H OK 7155 KB
OHXPER_H OK 18600 KB


In [15]:
import pandas as pd

# Load the five files
demo = pd.read_sas("DEMO_H.xpt",   format="xport")
ghb  = pd.read_sas("GHB_H.xpt",    format="xport")
bmx  = pd.read_sas("BMX_H.xpt",    format="xport")
den  = pd.read_sas("OHXDEN_H.xpt", format="xport")
per  = pd.read_sas("OHXPER_H.xpt", format="xport")

# How big is each file? (rows = people, columns = variables)
for name, d in [("demo", demo), ("ghb", ghb), ("bmx", bmx), ("den", den), ("per", per)]:
    print(name, d.shape)

# Merge age, sex, HbA1c and BMI using SEQN (the participant ID)
df = demo[["SEQN", "RIDAGEYR", "RIAGENDR"]]
df = df.merge(ghb[["SEQN", "LBXGH"]], on="SEQN", how="inner")
df = df.merge(bmx[["SEQN", "BMXBMI"]], on="SEQN", how="inner")

print("\nMerged table:", df.shape)
print(df.head())
print("\nMissing values:")
print(df.isna().sum())

# Look at the oral health variables (we will choose from these next)
print("\nDentition columns:", list(den.columns))
print("\nPeriodontal columns:", list(per.columns)[:40])

demo (10175, 47)
ghb (6979, 2)
bmx (9813, 26)
den (9422, 110)
per (4669, 508)

Merged table: (6979, 5)
      SEQN  RIDAGEYR  RIAGENDR  LBXGH  BMXBMI
0  73557.0      69.0       1.0   13.9    26.7
1  73558.0      54.0       1.0    9.1    28.6
2  73559.0      72.0       1.0    8.9    28.9
3  73561.0      73.0       2.0    4.9    19.7
4  73562.0      56.0       1.0    5.5    41.7

Missing values:
SEQN          0
RIDAGEYR      0
RIAGENDR      0
LBXGH       336
BMXBMI       86
dtype: int64

Dentition columns: ['SEQN', 'OHDEXSTS', 'OHDDESTS', 'OHXIMP', 'OHX01TC', 'OHX02TC', 'OHX03TC', 'OHX04TC', 'OHX05TC', 'OHX06TC', 'OHX07TC', 'OHX08TC', 'OHX09TC', 'OHX10TC', 'OHX11TC', 'OHX12TC', 'OHX13TC', 'OHX14TC', 'OHX15TC', 'OHX16TC', 'OHX17TC', 'OHX18TC', 'OHX19TC', 'OHX20TC', 'OHX21TC', 'OHX22TC', 'OHX23TC', 'OHX24TC', 'OHX25TC', 'OHX26TC', 'OHX27TC', 'OHX28TC', 'OHX29TC', 'OHX30TC', 'OHX31TC', 'OHX32TC', 'OHX02CTC', 'OHX03CTC', 'OHX04CTC', 'OHX05CTC', 'OHX06CTC', 'OHX07CTC', 'OHX08CTC', 'OHX09CTC', 

## 2. Define the outcome
Diabetes = HbA1c of 6.5% or higher. People with no HbA1c result are removed.
HbA1c is not used as a predictor, because it defines the outcome (data leakage).

In [16]:
df = df.dropna(subset=["LBXGH"]).copy()          # keep only people with an HbA1c result
df["diabetes"] = (df["LBXGH"] >= 6.5).astype(int)  # 1 = diabetes, 0 = no diabetes
print(df.shape)
print(df.columns.tolist())

(6643, 6)
['SEQN', 'RIDAGEYR', 'RIAGENDR', 'LBXGH', 'BMXBMI', 'diabetes']


## 3. Create oral health variables
For 28 teeth (third molars excluded) I counted: natural teeth present,
teeth lost to dental disease, and teeth with untreated decay.
Codes were checked against the CDC codebook. People whose dental exam was
not done are set to missing, not zero. Only adults aged 30+ are kept.

In [17]:
import numpy as np

# 28 teeth (third molars #1, 16, 17, 32 excluded)
teeth = [t for t in range(2, 32) if t not in (16, 17)]

def to_text(col):
    # CDC letter codes come in as bytes, e.g. b'S' -> 'S'
    return den[col].apply(lambda x: x.decode().strip() if isinstance(x, bytes) else x)

tc  = den[[f"OHX{t:02d}TC" for t in teeth]]
ctc = pd.DataFrame({t: to_text(f"OHX{t:02d}CTC") for t in teeth})
csc = pd.DataFrame({t: to_text(f"OHX{t:02d}CSC") for t in teeth})

feat = pd.DataFrame({"SEQN": den["SEQN"]})
feat["n_teeth"] = (tc == 2).sum(axis=1)                          # natural teeth present (max 28)
feat["n_missing_disease"] = ctc.isin(["E", "P", "R"]).sum(axis=1)  # lost to dental disease

# Decayed = tooth coded Z whose surface code contains a caries digit (0-4)
has_caries = csc.apply(lambda s: s.astype(str).str.contains("[0-4]", regex=True))
feat["n_decayed"] = ((ctc == "Z") & has_caries).sum(axis=1)

# If the dentition exam was not done, the counts are unknown, not zero
done = den["OHDDESTS"] == 1
feat.loc[~done, ["n_teeth", "n_missing_disease", "n_decayed"]] = np.nan

# Join to the diabetes table and keep adults 30+
data = df.merge(feat, on="SEQN", how="inner")
data = data[data["RIDAGEYR"] >= 30].copy()

print(data.shape)
print(data[["n_teeth", "n_missing_disease", "n_decayed"]].describe())
print(data.groupby("diabetes")[["n_teeth", "n_missing_disease", "n_decayed"]].mean().round(2))

(4502, 9)
           n_teeth  n_missing_disease    n_decayed
count  4183.000000        4183.000000  4183.000000
mean     20.758786           6.346880     0.904853
std       8.844526           8.628882     2.138225
min       0.000000           0.000000     0.000000
25%      18.000000           0.000000     0.000000
50%      24.000000           2.000000     0.000000
75%      27.000000           9.000000     1.000000
max      28.000000          28.000000    21.000000
          n_teeth  n_missing_disease  n_decayed
diabetes                                       
0           21.22               5.92       0.92
1           17.64               9.22       0.82


In [22]:
# 1. Define diabetes from HbA1c (6.5% or higher), after removing missing HbA1c
df = df.dropna(subset=["LBXGH"]).copy()
df["diabetes"] = (df["LBXGH"] >= 6.5).astype(int)
print("People with HbA1c:", df.shape[0])
print("Diabetes (all ages):", df["diabetes"].mean().round(3))

# 2. Who had a periodontal exam? Check the age range
pm = per.merge(demo[["SEQN", "RIDAGEYR"]], on="SEQN")
print("\nAge of people in periodontal file:")
print(pm["RIDAGEYR"].describe())
print("\nPeriodontal exam status codes:")
print(per["OHDPDSTS"].value_counts())

# 3. Adults only: diabetes frequency by age group
adults = df[df["RIDAGEYR"] >= 30]
print("\nAdults 30+:", adults.shape[0])
print(adults["diabetes"].value_counts())
print("Diabetes (adults 30+):", adults["diabetes"].mean().round(3))

# 4. Look at the codes used in the dentition file
print("\nDentition exam status:")
print(den["OHDDESTS"].value_counts())
print("\nTooth status, tooth 2 (OHX02TC):")
print(den["OHX02TC"].value_counts())
print("\nCoronal caries, tooth 2 (OHX02CTC):")
print(den["OHX02CTC"].value_counts())

People with HbA1c: 6643
Diabetes (all ages): 0.091

Age of people in periodontal file:
count    4669.000000
mean       54.047548
std        14.811855
min        30.000000
25%        41.000000
50%        53.000000
75%        66.000000
max        80.000000
Name: RIDAGEYR, dtype: float64

Periodontal exam status codes:
OHDPDSTS
1.0    3982
3.0     611
2.0      76
Name: count, dtype: int64

Adults 30+: 4502
diabetes
0    3908
1     594
Name: count, dtype: int64
Diabetes (adults 30+): 0.132

Dentition exam status:
OHDDESTS
1.0    8964
3.0     456
2.0       2
Name: count, dtype: int64

Tooth status, tooth 2 (OHX02TC):
OHX02TC
2.0    5037
4.0    3844
5.0      81
3.0       6
Name: count, dtype: int64

Coronal caries, tooth 2 (OHX02CTC):
OHX02CTC
b'S'    2637
b'U'    2401
b'Z'    2375
b'E'     745
b'P'     644
b''      457
b'J'      81
b'Q'      34
b'R'      22
b'Y'      22
b'M'       3
b'X'       1
Name: count, dtype: int64


## Check for confounding
People with diabetes were older on average (60.8 vs 53.0 years), so age
could explain part of the link with tooth loss. Age is therefore included
in the model.

In [18]:
print(data.groupby("diabetes")["RIDAGEYR"].mean().round(1))

diabetes
0    53.0
1    60.8
Name: RIDAGEYR, dtype: float64


## 4. Split into training and test sets
80% of people are used to train the model and 20% are kept for testing.
stratify keeps the share of people with diabetes the same in both groups.
random_state makes the split reproducible.

In [19]:
from sklearn.model_selection import train_test_split

cols = ["RIDAGEYR", "RIAGENDR", "BMXBMI", "n_teeth", "n_missing_disease", "n_decayed"]
model_data = data[cols + ["diabetes"]].dropna()
print("Rows before:", data.shape[0], " Rows after dropping missing:", model_data.shape[0])

X = model_data[cols]
y = model_data["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training people:", X_train.shape[0], " Test people:", X_test.shape[0])
print("Diabetes in training:", y_train.mean().round(3))
print("Diabetes in test:", y_test.mean().round(3))

Rows before: 4502  Rows after dropping missing: 4150
Training people: 3320  Test people: 830
Diabetes in training: 0.127
Diabetes in test: 0.127


## 5. Train and test a logistic regression model
The model learns from the training people only, then predicts risk for the
test people, who it has not seen. AUC measures how well it ranks people by risk.
Sensitivity and specificity depend on the cut-off: at 0.5 the model caught
almost no cases, so I also tried a cut-off equal to the overall prevalence (0.127).

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

# Create and train the model (training people only)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predicted risk (0 to 1) for each person
train_probs = model.predict_proba(X_train)[:, 1]
test_probs  = model.predict_proba(X_test)[:, 1]

print("AUC on training people:", round(roc_auc_score(y_train, train_probs), 3))
print("AUC on test people:    ", round(roc_auc_score(y_test, test_probs), 3))

# Turn risks into yes/no predictions at two cut-offs
for threshold in [0.5, 0.127]:
    pred = (test_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"\nCut-off {threshold}:")
    print("  Sensitivity:", round(tp / (tp + fn), 3))
    print("  Specificity:", round(tn / (tn + fp), 3))

AUC on training people: 0.723
AUC on test people:     0.774

Cut-off 0.5:
  Sensitivity: 0.029
  Specificity: 0.997

Cut-off 0.127:
  Sensitivity: 0.771
  Specificity: 0.669


## 6. Do oral health variables add anything? (5-fold cross-validation)
Cross-validation trains and tests the model five times on different parts of the
data and averages the results, which is more reliable than one split.
Two models are compared: age, sex and BMI only, versus the same plus oral
health variables. Odds ratios are shown for the full model.

In [21]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

base_cols = ["RIDAGEYR", "RIAGENDR", "BMXBMI"]
oral_cols = ["n_teeth", "n_missing_disease", "n_decayed"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, c in [("Age, sex, BMI only", base_cols),
                ("Age, sex, BMI + oral health", base_cols + oral_cols)]:
    scores = cross_val_score(LogisticRegression(max_iter=1000), X[c], y,
                             cv=cv, scoring="roc_auc")
    print(f"{name}: mean AUC {scores.mean():.3f} (SD {scores.std():.3f})")

# Odds ratios from the full model (per 1-unit increase)
or_table = pd.DataFrame({"predictor": cols,
                         "odds_ratio": np.exp(model.coef_[0]).round(3)})
print(or_table)

Age, sex, BMI only: mean AUC 0.726 (SD 0.024)
Age, sex, BMI + oral health: mean AUC 0.731 (SD 0.024)
           predictor  odds_ratio
0           RIDAGEYR       1.035
1           RIAGENDR       0.585
2             BMXBMI       1.076
3            n_teeth       0.966
4  n_missing_disease       0.984
5          n_decayed       1.005
